# 03 — LoRA desde cero: entrenar el 1 % y dejar el 99 % congelado

**Pregunta que responde este notebook:** ¿cómo puede un modelo de 7.000 millones de parámetros adaptarse a una tarea nueva entrenando solo unos pocos millones?

La respuesta es **LoRA** (*Low-Rank Adaptation*), implementado aquí desde cero en `src/skillgen/training/fundamentals.py` (clase `LoRALinear`, con sus tests en `tests/test_lora.py`). Nada de PEFT todavía: primero la idea desnuda, después la librería.

La fórmula completa es una sola línea:

$$W' = W + \Delta W, \qquad \Delta W = \frac{\alpha}{r}\, B A$$

- $W$ es la matriz original, **congelada**: nunca recibe gradiente.
- $A$ tiene forma $(r, \text{in})$ y $B$ forma $(\text{out}, r)$, con un rango $r$ pequeño (8, 16…).
- Solo $A$ y $B$ se entrenan: $r \times (\text{in} + \text{out})$ parámetros en vez de $\text{in} \times \text{out}$.
- $B$ se inicializa a **cero**, así que al principio $\Delta W = 0$ y el modelo se comporta *exactamente* igual que el original. $A$ se inicializa con kaiming para romper la simetría.

## 1. Envolver capas de un MLP pequeño

Tomamos un MLP de juguete y sustituimos sus dos `nn.Linear` por `LoRALinear`. Lo primero es mirar los números reales: cuántos parámetros hay en total y cuántos entrenan de verdad.

In [1]:
import sys
from pathlib import Path

import torch
from torch import nn

torch.manual_seed(0)

# Localizar la raíz del repo subiendo desde el directorio del notebook.
root = Path.cwd()
while not (root / "data" / "splits").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

from skillgen.training.fundamentals import LoRALinear, lora_parameter_count


class ToyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = LoRALinear(nn.Linear(20, 64), r=4, alpha=8)
        self.fc2 = LoRALinear(nn.Linear(64, 3), r=4, alpha=8)
        self.act = nn.GELU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


model = ToyMLP()
print(model.fc1)
print(model.fc2)
print()
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parámetros totales:     {total:,}")
print(f"parámetros entrenables: {trainable:,}  ({100 * trainable / total:.1f} %)")
print(f"comprobación con la fórmula r*(in+out): "
      f"{lora_parameter_count(20, 64, 4) + lora_parameter_count(64, 3, 4):,}")

LoRALinear(
  in_features=20, out_features=64, r=4, alpha=8, scaling=2.0
  (base): Linear(in_features=20, out_features=64, bias=True)
  (dropout): Identity()
)
LoRALinear(
  in_features=64, out_features=3, r=4, alpha=8, scaling=2.0
  (base): Linear(in_features=64, out_features=3, bias=True)
  (dropout): Identity()
)

parámetros totales:     2,143
parámetros entrenables: 604  (28.2 %)
comprobación con la fórmula r*(in+out): 604


**Ojo con el porcentaje:** en este MLP de juguete LoRA "solo" ahorra hasta dejar ~28 % entrenable, porque las capas son diminutas y el rango (4) no es tan pequeño en comparación. La magia aparece en capas grandes. En una capa cuadrada típica de un modelo de 7B (3584 × 3584), el contraste es brutal:

In [2]:
in_f = out_f = 3584  # ancho típico de Qwen2.5-7B
full = in_f * out_f
for r in (8, 16, 32):
    lora = lora_parameter_count(in_f, out_f, r)
    print(f"r={r:2d}: capa completa {full:>12,} params | LoRA {lora:>9,} params "
          f"({100 * lora / full:.2f} % entrenable)")

r= 8: capa completa   12,845,056 params | LoRA    57,344 params (0.45 % entrenable)
r=16: capa completa   12,845,056 params | LoRA   114,688 params (0.89 % entrenable)
r=32: capa completa   12,845,056 params | LoRA   229,376 params (1.79 % entrenable)


## 2. La cuenta de VRAM: por qué QLoRA cabe en una T4 de 16 GB

Entrenar no cuesta solo los pesos. En un *fine-tuning* completo con Adam hay que guardar, **por parámetro**:

| Componente | Bytes/parámetro (fp16 + Adam clásico) |
|---|---|
| Pesos | 2 |
| Gradientes | 2 |
| Estados de Adam (dos momentos, fp32) | 8 |
| **Total** | **12** |

**QLoRA** cambia las reglas: el modelo base se guarda **cuantizado a 4 bits** (0,5 bytes/parámetro) y congelado; solo los adaptadores LoRA viven en fp16 y solo ellos pagan optimizador (en su variante de 8 bits: dos estados de 1 byte por parámetro del adaptador).

Las dos cuentas, hechas en código para un modelo de 7B con adaptadores r=16 sobre las 7 proyecciones lineales de cada una de sus 28 capas (dimensiones tipo Qwen2.5-7B: oculto 3584, intermedio 18944, atención agrupada con KV de 512):

In [3]:
P = 7_000_000_000  # 7B parámetros

# --- Fine-tuning completo: pesos + gradientes + Adam, todo por parámetro ---
full_bytes = P * (2 + 2 + 8)

# --- QLoRA: base a 4 bits + adaptadores fp16 + optimizador de 8 bits ---
per_layer_adapters = sum([
    lora_parameter_count(3584, 3584, 16),   # q
    lora_parameter_count(3584, 512, 16),    # k (GQA)
    lora_parameter_count(3584, 512, 16),    # v (GQA)
    lora_parameter_count(3584, 3584, 16),   # o
    lora_parameter_count(3584, 18944, 16),  # gate
    lora_parameter_count(3584, 18944, 16),  # up
    lora_parameter_count(18944, 3584, 16),  # down
])
adapter_params = per_layer_adapters * 28  # 28 capas
qlora_bytes = P * 0.5 + adapter_params * 2 + adapter_params * 2

GB = 1e9
print(f"adaptadores LoRA totales: {adapter_params:,} parámetros "
      f"({100 * adapter_params / P:.2f} % del modelo)")
print()
print(f"Fine-tuning completo: {full_bytes / GB:6.2f} GB")
print(f"QLoRA:                 {qlora_bytes / GB:6.2f} GB")
print(f"Reducción:             {full_bytes / qlora_bytes:.1f}× menos memoria")

adaptadores LoRA totales: 40,370,176 parámetros (0.58 % del modelo)

Fine-tuning completo:  84.00 GB
QLoRA:                   3.66 GB
Reducción:             22.9× menos memoria


La lectura que importa: el *fine-tuning* completo necesita del orden de **84 GB** solo en estados de entrenamiento —imposible en una T4 de 16 GB— mientras que QLoRA ronda los **3,7 GB** en la misma cuenta simplificada (en la práctica hay que sumar activaciones y el propio lote, pero sigue cabiendo con margen). No es una optimización cosmética: es la diferencia entre "no se puede" y "se puede esta noche en Kaggle gratis".

Aviso honesto: esta es la cuenta de los **estados** (pesos, gradientes, optimizador). Las activaciones dependen del tamaño de lote y de la longitud de secuencia —por eso la Fase 0 fijó `max_seq_length = 8192` con reglas de *fallback*— pero no cambian la conclusión.

## 3. Smoke test: 10 pasos entrenando SOLO LoRA

Prueba mínima de que el mecanismo funciona de punta a punta:

1. Creamos una tarea de clasificación sintética (3 clases) con etiquetas generadas por una proyección fija.
2. Guardamos una **copia exacta** de los pesos base antes de entrenar.
3. Entrenamos 10 pasos pasando al optimizador **únicamente** los parámetros con `requires_grad=True` (es decir, los $A$ y $B$).
4. Comprobamos dos cosas: que la pérdida **baja** y que los pesos base son **bit a bit idénticos** después de entrenar (`torch.equal`, no "casi iguales").

In [4]:
# Datos sintéticos: las etiquetas salen de una proyección lineal fija.
X = torch.randn(128, 20)
teacher = nn.Linear(20, 3, bias=False)
with torch.no_grad():
    y = (X @ teacher.weight.T).argmax(dim=1)

# Copia exacta de los pesos base ANTES de entrenar.
base_before = [model.fc1.base.weight.clone(), model.fc2.base.weight.clone()]

# El optimizador recibe SOLO parámetros entrenables (los adaptadores LoRA).
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=0.05
)
loss_fn = nn.CrossEntropyLoss()

for step in range(11):
    loss = loss_fn(model(X), y)
    print(f"paso {step:2d}   loss {loss.item():.4f}")
    if step == 10:
        break
    loss.backward()      # gradiente solo para A y B (la base está congelada)
    optimizer.step()
    optimizer.zero_grad()

# La base NO se ha movido: igualdad bit a bit, no aproximada.
assert torch.equal(base_before[0], model.fc1.base.weight)
assert torch.equal(base_before[1], model.fc2.base.weight)
print()
print("pesos base idénticos antes/después: True (torch.equal)")

paso  0   loss 1.1446
paso  1   loss 1.0716
paso  2   loss 0.8925
paso  3   loss 0.6658
paso  4   loss 0.4587
paso  5   loss 0.3097
paso  6   loss 0.1952
paso  7   loss 0.1175
paso  8   loss 0.0704
paso  9   loss 0.0358
paso 10   loss 0.0180

pesos base idénticos antes/después: True (torch.equal)


## 4. Cierre: por qué la Fase 3 usa QLoRA y no fine-tuning completo

Juntando las tres piezas de este notebook:

1. **LoRA funciona**: 10 pasos bastan para que la pérdida caiga entrenando solo los adaptadores, y la base queda intacta — el conocimiento general del modelo no se toca, se le *añade* una corrección de rango bajo.
2. **La memoria manda**: una T4 de Kaggle tiene 16 GB. El fine-tuning completo de un 7B necesita ~84 GB de estados; QLoRA, ~3,7 GB en la misma cuenta. No hay decisión que tomar: en nuestro hardware, QLoRA es la única vía.
3. **El adaptador es un artefacto pequeño y portable**: decenas de millones de parámetros (cientos de MB) en vez de un modelo completo. Se puede versionar, compartir y aplicar sobre la base cuantizada — y al final del todo (Fase 5) fusionarlo (*merge*) y exportar.

En la Fase 3, PEFT hará exactamente lo que hace `LoRALinear` aquí —envolver capas, congelar la base, entrenar $A$ y $B$— pero sobre el modelo real cuantizado a 4 bits con `bitsandbytes`. Si entiendes este notebook, PEFT deja de ser magia: es este mismo código, industrializado.